In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import sionna.rt as rt
from utils.scene_utils import add_txs, set_frequence
from utils.material_factory import create_sionna_material
from utils.geo_coords import SceneCoordinateConverter
from config import FILE_ENCODE, LocalCRS

# Define the geographical boundary of the target region:
# I take the max and min latitude and longitude of transmitters and extend it with 500m
# Extended Formula:
# latitude 1° is approximately equal to 111,000m
# 500 / 111 000 ≈ 0.004505
# 48.90138888888889 + 0.004505 ≈ 48.9059; 48.818333333333335 - 0.004505 ≈ 48.8138
# longitude 1° is approximately equal to $\Delta_{lon} =\Delta_{lat} \times \cos(lat)$
# 111 000 * cos((48.9014 + 48.8183)/2) ≈ 111 000 * cos(48.86) ≈ 111 000 * 0.6579 ≈ 73 000
# 500 / 73 000 ≈ 0.006849
# 2.249722222222222 - 0.006849 ≈ 2.2429 ; 2.450555555555556 + 0.006849 ≈ 2.4574
LAT_MAX, LAT_MIN = 48.9059, 48.8138
LON_MIN, LON_MAX = 2.2429, 2.4574

# Calculate the center origin point of the scene
LAT_ORIGIN=(LAT_MAX + LAT_MIN)/2
LON_ORIGIN=(LON_MIN + LON_MAX)/2
TRANSMITTER_DIRECTORY = Path('data/transmitters')

In [ ]:
# Optional: Preprocess antenna files. Filter antennas by frequence and postal code
from utils.preprocess_raw_data import preprocess_antenna_data_by_frequency_and_postal

ANTENNES_INFO_FILENAME = Path('data/paris_whole/Antennes_Emetteurs_Bandes_Cartoradio.csv')
ANTENNES_LOC_FILENAME = Path('data/paris_whole/Sites_Cartoradio.csv')
FILTER_FREQUENCE = 2600
FILTER_POSTAL_CODE = r'^75\d{3}$'

preprocess_antenna_data_by_frequency_and_postal(
    ANTENNES_INFO_FILENAME, 
    ANTENNES_LOC_FILENAME, 
    FILTER_FREQUENCE, 
    FILTER_POSTAL_CODE, 
    TRANSMITTER_DIRECTORY)

# Step 1: Region Grid Splitting
In this step, we split the large Paris area into smaller blocks (e.g., 2.56km x 2.56km). 
This helps our computer run Sionna simulation without running out of memory.

**Key features:**
* Use official French national projection - **Lambert-93** to work with meters.
* Add an **overlap buffer** (e.g., 150m) to avoid edge errors in ray-tracing.

In [ ]:
# Import the splitter class from map_splitter.py
from utils.map_splitter import TileSplitter

# Initialize the splitter
# Each block is 2560m x 2560m. Overlap is 150m.
splitter = TileSplitter(
    lat_min=LAT_MIN, 
    lat_max=LAT_MAX, 
    lon_min=LON_MIN, 
    lon_max=LON_MAX, 
    block_size_m=2560, 
    overlap_m=150
)

# Get the task list for loops later
all_blocks = splitter.get_all_blocks()